# Detecção de atipicidade contábil em instituições financeiras S2 e S3**Rede neural sobre dados públicos do IF.data/COSIF — Banco Central do Brasil**Apoio ao TCC *Auditorias em instituições financeiras dos segmentos S2 e S3*(Universidade Federal Fluminense).---## Antes de executar: o que este notebook faz e o que ele não faz**Faz.** Coleta demonstrações trimestrais públicas de instituições dos segmentos S2 e S3,constrói indicadores de risco de manipulação adaptados ao balanço bancário, treina uma redeneural que aprende o padrão de comportamento normal do conjunto, e ordena as observações(instituição × trimestre) por distância a esse padrão.**Não faz.** Não detecta fraude. O que se mede é *atipicidade contábil*. Fraude é qualificaçãojurídica: exige intenção comprovada, contraditório e decisão de autoridade competente. Um escorealto significa "esta instituição se comportou de modo diferente dos seus pares e do seu própriohistórico" — o que é hipótese para direcionar procedimento de auditoria, não achado.Essa distinção não é formalidade. Dyck, Morse e Zingales (2020) estimam que apenas cerca de umterço das fraudes corporativas é detectado; logo, o conjunto de instituições *sem* evento públicoconhecido contém tanto instituições íntegras quanto instituições cuja irregularidade nunca veio apúblico. Nenhum modelo treinado sobre esses dados consegue separar as duas coisas, e nenhumresultado deste notebook deve ser redigido como se conseguisse.## Estrutura| Etapa | Seção | O que é decidido ali ||---|---|---|| Descoberta do esquema | 1–2 | Que recursos o IF.data realmente publica || Coleta e segmentação | 3–4 | Quem entra na amostra S2/S3 || Mapeamento contábil | 5 | Qual conta COSIF corresponde a cada conceito || Indicadores adaptados | 6 | Como Beneish/Dechow/Piotroski viram indicadores bancários || Painel e partição | 7–8 | Tratamento, corte temporal e auditoria de vazamento || Redes neurais | 9–10 | Autoencoder (transversal) e GRU (temporal) || Comparação | 11, 15 | A rede acrescenta algo sobre PCA, Mahalanobis, XGBoost/LightGBM? || Interpretação e evidência | 12–14 | Por que a instituição é atípica; triangulação; eventos públicos || **Pontos de correção** | **16** | **Tudo que ficou fora de alcance, com a área responsável** |

---## 1. Ambiente e configuraçãoTodos os parâmetros metodológicos estão em `src/fraude_s2s3/config.py`, fora do código deexecução, para que possam ser auditados sem leitura de implementação.

### 1.1 Recarregamento automatico dos modulosAs duas linhas magicas abaixo fazem o notebook reler os arquivos de `src/`sempre que uma celula e executada.Sem elas, o Python mantem em cache o modulo ja importado: voce edita`src/fraude_s2s3/ifdata.py`, reexecuta a celula, e ela continua rodando ocodigo antigo — **sem erro e sem aviso**. Como o trabalho aqui e iterativo(ajustar a coleta, reavaliar as metricas, ajustar de novo), essa e a primeiracoisa que precisa estar ligada.Limite conhecido: o `autoreload` nao recarrega objetos ja instanciados nemalteracoes em `@dataclass` de configuracao. Se mexer em `config.py`, reinicieo kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, warnings
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))          # processar_base.py fica na raiz
sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "testes"))
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fraude_s2s3 import (avaliacao, baselines, benford, diagnostico, ifdata,
                         indicadores, interpretacao, mapeamento, painel,
                         redes, segmentacao, testes_analiticos, vazamento)
from fraude_s2s3.config import CONFIG, DIR_EXTERNOS, DIR_RESULTADOS, periodos

pd.set_option("display.width", 160, "display.max_columns", 40)
diagnostico.limpar()

print("Kernel em uso:", sys.executable)
print("  ^ Deve ser o MESMO Python onde voce rodou 'python -m pip install'.")
print("    Para conferir, rode no PowerShell:  python -c \"import sys; print(sys.executable)\"")
print("    Se divergir: Select Kernel (canto superior direito) e escolha o caminho certo.\n")

print("Recorte:", periodos()[0], "a", periodos()[-1], f"({len(periodos())} trimestres)")
print("Segmentos:", CONFIG.__class__.__module__, "->", "S2 e S3")
print("Corte de treino:", CONFIG.avaliacao.corte_treino, "| validacao ate:", CONFIG.avaliacao.corte_validacao)
print("PyTorch disponivel:", redes.TORCH_DISPONIVEL)

### 1.2 Chave de execução: dados reais ou teste de fumaça`USAR_DADOS_REAIS = True` executa contra a API do Banco Central.`False` roda com um painel **artificial** (`testes/fixture_fumaca.py`), cuja única finalidade éverificar se o código executa de ponta a ponta. Números produzidos nesse modo **não podemaparecer no TCC** — não são dados, são ruído pseudoaleatório com o formato certo.

In [ ]:
USAR_DADOS_REAIS = True   # <- alterne para False apenas para testar a execucao

if not USAR_DADOS_REAIS:
    print("=" * 88)
    print("MODO TESTE DE FUMACA — PAINEL ARTIFICIAL".center(88))
    print("Nenhum numero desta execucao pode ser usado no TCC.".center(88))
    print("=" * 88)

---
## 2. Origem dos dados

O IF.data publica os mesmos dados por duas vias: o serviço OData (`olinda.bcb.gov.br`) e a
exportação manual pela interface web. **Esta pesquisa usa a exportação manual**, por duas razões
registradas no histórico: a rede do projeto não alcança `bcb.gov.br`, e o arquivo exportado é
fonte imutável, anexável ao trabalho e conferível pela banca.

A célula abaixo decide a rota sozinha, nesta ordem:

| Rota | Condição | O que faz |
|---|---|---|
| `tratado` | existe `dados/tratados/painel_indicadores_S2S3.csv` | carrega o painel pronto e segue para a seção 7 |
| `local` | existe CSV em `dados/ifdata_bruto/` | processa a base e grava o painel |
| `api` | não há base local | tenta o serviço OData (seções 2.1 e 3.1) |
| `fumaca` | `USAR_DADOS_REAIS = False` | painel artificial, só para testar a execução |

**A rota `api` é a última, não a primeira.** Se você tem a base baixada, a descoberta de esquema
não é executada e a falha de rede não aparece — ela não é um erro a corrigir, é uma via que este
projeto não usa.

In [ ]:
DIR_BRUTO   = RAIZ / "dados" / "ifdata_bruto"
DIR_TRATADO = RAIZ / "dados" / "tratados"
PAINEL_IND  = DIR_TRATADO / "painel_indicadores_S2S3.csv"
PAINEL_CONC = DIR_TRATADO / "painel_conceitos_S2S3.csv"

if not USAR_DADOS_REAIS:
    ROTA = "fumaca"
elif PAINEL_IND.exists():
    ROTA = "tratado"
elif any(DIR_BRUTO.rglob("*.csv")):
    ROTA = "local"
else:
    ROTA = "api"

print(f"ROTA = {ROTA}\n")

if ROTA == "tratado":
    print("Painel ja processado encontrado:")
    print(f"    {PAINEL_IND.relative_to(RAIZ)}")
    print("\nAs secoes 2.1 a 6 sao puladas — o painel ja passou por elas.")
    print("Siga para a secao 7. Para reprocessar do zero, apague a pasta dados/tratados.")
elif ROTA == "local":
    n = len(list(DIR_BRUTO.rglob("*.csv")))
    print(f"Base bruta encontrada: {n} arquivos CSV em {DIR_BRUTO.relative_to(RAIZ)}")
    print("\nA secao 3.2 processa a base e grava dados/tratados.")
elif ROTA == "api":
    print("Nenhuma base local. Tentando o servico OData do Bacen.\n")
    cliente = ifdata.ClienteIFData()
    try:
        catalogo_servico = cliente.descrever_servico()
        display(catalogo_servico)
        print("\nPeriodos publicados (amostra):", cliente.periodos_disponiveis()[:8], "...")
    except ifdata.ErroColeta as e:
        print("O servico OData nao respondeu:\n ", e)
        print("\n" + "=" * 78)
        print("ISTO NAO E UM DEFEITO DO CODIGO.")
        print("=" * 78)
        print("A rota prevista para esta pesquisa e a exportacao manual.")
        print("\nO que fazer:")
        print("  1. Exportar os relatorios pela interface web do IF.data")
        print("     (ver docs/GUIA_EXTRACAO_MANUAL.md)")
        print(f"  2. Salvar em {DIR_BRUTO.relative_to(RAIZ)}, na estrutura ano/anomes/")
        print("  3. Executar esta celula de novo — a rota passa a ser 'local'")

### 2.1 Escolha dos relatórios a coletar — **somente na rota `api`**

Se a rota for `tratado` ou `local`, pule para a seção 3.2: a escolha de relatórios já está feita
em `consolidacao.py`, que mapeia explicitamente 52 conceitos contábeis sobre os seis relatórios
exportados.

> **Decisão registrada no TCC:** a unidade é o **conglomerado prudencial**, não a instituição
> individual. É sobre o conglomerado que incide o requerimento prudencial e sobre ele que a
> Resolução CMN 4.553/2017 segmenta. A escolha muda quem é S2 e quem é S3, e por isso precisa
> estar explícita no texto.

In [ ]:
RELATORIOS = []          # ex.: ["1", "2", "4"] — confirmar na tabela da secao 2
TIPOS_INSTITUICAO = []   # ex.: [2] — confirmar na tabela da secao 2

if ROTA == "api" and not RELATORIOS:
    diagnostico.registrar(
        area="notebook/seccao-2",
        item="RELATORIOS e TIPOS_INSTITUICAO nao foram preenchidos.",
        impacto="A coleta pela API nao pode ser executada; o painel fica vazio.",
        acao=("Rodar a celula da secao 2, ler a tabela de recursos publicados e preencher os "
              "codigos de relatorio e de tipo de instituicao. Area: notebook, secao 2.1."),
        severidade=diagnostico.Severidade.BLOQUEIA,
    )
    print("PENDENTE: preencher RELATORIOS e TIPOS_INSTITUICAO (ver secao 16).")
else:
    print(f"Rota {ROTA}: secao 2.1 nao se aplica.")

---## 3. Coleta### 3.1 Via APIFalhas isoladas (um trimestre, um relatório) não interrompem a varredura: são registradas e operíodo fica ausente no painel — o que a seção 16 reporta. Interromper tudo por uma ausênciatornaria a coleta impraticável; escondê-la tornaria o painel mentiroso.

In [ ]:
if ROTA == "api" and RELATORIOS and TIPOS_INSTITUICAO:
    bruto_longo = cliente.coletar_painel(RELATORIOS, TIPOS_INSTITUICAO)
    print("Registros coletados:", f"{len(bruto_longo):,}")
    print("Coletado em:", bruto_longo.attrs.get("coletado_em"), "<- CITAR ESTA DATA NO TCC")
    display(bruto_longo.head())
elif ROTA == "fumaca":
    from fixture_fumaca import painel_artificial
    bruto_largo = painel_artificial(n_instituicoes=60)
    print(bruto_largo.attrs["AVISO"])
else:
    print(f"Rota {ROTA}: secao 3.1 nao se aplica.")

### 3.2 Rota local — exportações manuais

**É esta a rota desta pesquisa.** Os CSV exportados ficam em `dados/ifdata_bruto/`, organizados
por ano e trimestre:

```
dados/ifdata_bruto/IF DATA BASE/2019/201903/dados 2019-03 Ativo.csv
                                /201906/...
```

`consolidacao.consolidar()` lê os seis relatórios de cada trimestre e monta o painel de conceitos
contábeis; `segmentacao` aplica o recorte S2/S3 pela coluna `SR` do próprio relatório; e
`indicadores.construir()` produz os indicadores. A célula abaixo executa os três via
`processar_base.py`, que é o mesmo caminho do script de linha de comando — não há lógica
duplicada entre notebook e script.

In [ ]:
import pandas as pd

if ROTA == "local":
    # Mesmo caminho do script de linha de comando, para nao haver divergencia
    # entre o que o notebook faz e o que processar_base.py faz.
    _argv = sys.argv
    sys.argv = ["processar_base.py"]   # Jupyter injeta args do kernel em sys.argv
    try:
        import processar_base
        processar_base.main()
    finally:
        sys.argv = _argv
    if PAINEL_IND.exists():
        ROTA = "tratado"
        print(f"\n\nROTA agora = {ROTA}")

if ROTA == "tratado":
    ind = pd.read_csv(PAINEL_IND)
    conceitos = pd.read_csv(PAINEL_CONC) if PAINEL_CONC.exists() else None
    print(f"Painel de indicadores: {ind.shape[0]:,} observacoes x {ind.shape[1]} colunas")
    print(f"Instituicoes: {ind['cod_inst'].nunique()}   Trimestres: {ind['anomes'].nunique()}")
    print("\nPor segmento:")
    print(ind.groupby("segmento")["cod_inst"].nunique().to_string())
    display(ind.head())

### 3.3 Pivotagem — **somente na rota `api`**

O serviço OData devolve formato longo (uma linha por instituição × conta × período) e os modelos
exigem formato largo. Na rota local isso já foi feito por `consolidacao.consolidar()`, que também
distingue soma de parcelas semânticas de coalescência de variantes de rótulo — distinção que uma
pivotagem ingênua erraria.

In [ ]:
if ROTA == "api" and RELATORIOS:
    COL_INST, COL_CONTA, COL_VALOR = "CodInst", "NomeColuna", "Saldo"   # <- confirmar
    faltando = [c for c in (COL_INST, COL_CONTA, COL_VALOR) if c not in bruto_longo.columns]
    if faltando:
        diagnostico.registrar(
            area="notebook/seccao-3.3",
            item=f"Colunas {faltando} nao existem no retorno da API. Disponiveis: {list(bruto_longo.columns)[:15]}",
            impacto="A pivotagem para painel largo nao pode ser feita com os nomes presumidos.",
            acao="Ajustar COL_INST/COL_CONTA/COL_VALOR aos nomes reais. Area: notebook, secao 3.3.",
            severidade=diagnostico.Severidade.BLOQUEIA,
        )
        print("PENDENTE:", faltando)
    else:
        bruto_largo = (bruto_longo
                       .pivot_table(index=[COL_INST, "anomes"], columns=COL_CONTA,
                                    values=COL_VALOR, aggfunc="first")
                       .reset_index().rename(columns={COL_INST: "cod_inst"}))
        print("Painel largo:", bruto_largo.shape)
else:
    print(f"Rota {ROTA}: secao 3.3 nao se aplica.")

---## 4. Segmentação S2 e S3Ordem de precedência:1. **Lista oficial de enquadramento** (`dados/externos/segmentacao_oficial.csv`) — única fonte que   reproduz a classificação do supervisor.2. **Reconstrução pela regra** Exposição/PIB da Resolução CMN 4.553/2017 — aproximação, porque a   norma considera exposições fora do balanço e porque o enquadramento é revisto em datas-base   próprias, não no trimestre em que a razão cruza o limiar.Se o caminho 2 for usado, o pipeline registra **pendência bloqueante**: o recorte amostral é aprimeira coisa que a banca vai questionar, e "reconstruí por regra" não é o mesmo que "usei aclassificação oficial".

In [ ]:
if ROTA == "tratado":
    print("Segmentacao ja aplicada no processamento (coluna SR dos proprios relatorios).")
    print(ind.groupby("segmento")["cod_inst"].agg(instituicoes="nunique", observacoes="size").to_string())
else:
    pib = segmentacao.carregar_pib()
    oficial = segmentacao.carregar_oficial()
    print("Lista oficial de segmentacao:", "encontrada" if oficial is not None else "AUSENTE")
    print("Serie de PIB trimestral:", "encontrada" if pib is not None else "AUSENTE")

    bruto_seg = segmentacao.aplicar(bruto_largo, pib) if (oficial is not None or pib is not None) else bruto_largo
    if "segmento" in bruto_seg.columns:
        display(bruto_seg.groupby("segmento").agg(instituicoes=("cod_inst", "nunique"),
                                                  observacoes=("cod_inst", "size")))

---## 5. Mapeamento contábilCada conceito é declarado por padrões de reconhecimento sobre o rótulo normalizado — não por nomeexato. Conceitos ambíguos são resolvidos por heurística **e reportados**; conceitos obrigatóriosausentes bloqueiam o pipeline.A tabela resultante é o *quadro de operacionalização das variáveis*: anexe-a ao TCC. É o documentoque permite a um terceiro reproduzir a construção dos indicadores a partir da mesma fonte.

In [ ]:
if ROTA == "tratado":
    print("Mapeamento contabil ja aplicado por consolidacao.py, que cobre 52 conceitos")
    print("sobre os seis relatorios. Ver dados/tratados/cobertura_conceitos.csv.")
elif USAR_DADOS_REAIS:
    rotulos = [c for c in bruto_seg.columns
               if c not in ("cod_inst", "anomes", "segmento", "segmento_periodo", "pib", "razao_exposicao_pib")]
    mapa = mapeamento.resolver(rotulos)
    print(f"Cobertura de conceitos: {mapa.cobertura:.0%}")
    display(mapa.tabela())
    print("\nRotulos do IF.data nao utilizados (candidatos a novos indicadores):")
    print(mapa.rotulos_nao_usados[:25])
    conceitos = mapeamento.aplicar(bruto_seg, mapa)
else:
    conceitos = bruto_seg

---## 6. Indicadores adaptados ao balanço bancárioEste é o ponto em que o trabalho se separa da simples replicação do artigo de referência.Beneish (1999), Dechow et al. (2011) e Piotroski (2000) foram calibrados em empresas **nãofinanceiras**. Em banco, parte dos componentes é inaplicável por construção: não há estoque, nãohá custo das mercadorias vendidas, "vendas" não é receita operacional no mesmo sentido, edepreciação é irrelevante frente ao tamanho do balanço. Aplicá-los sem adaptação produz número,não produz medida.A reconstrução é por **analogia funcional** — preserva a intenção econômica do indicador e troca aconta pelo equivalente bancário. Cada indicador declara sua classe de evidência:- `LIT` — correspondência direta na literatura citada;- `ADAP` — componente da literatura com substituição de conta;- `PROP` — proposição do autor a partir do arcabouço prudencial (Res. CMN 2.682/1999, Basileia).  **Sem validação empírica prévia**; é contribuição exploratória, sujeita à crítica da banca.

In [ ]:
display(indicadores.catalogo_df())
print("\n" + indicadores.alerta_evidencia())

In [ ]:
if ROTA == "tratado":
    print(f"Indicadores ja construidos: {ind.shape[1]} colunas.")
else:
    ind = indicadores.construir(conceitos)
    print("Construidos:", ind.attrs["indicadores_construidos"])
    print("\nAusentes  :", ind.attrs["indicadores_ausentes"] or "(nenhum)")

display(ind.describe().T[["count", "mean", "50%", "std", "min", "max"]].round(4))

---## 7. Painel: tratamento e partição temporalQuatro decisões materializadas aqui, todas discutíveis e todas explícitas:1. **Winsorização por trimestre**, não global — contas COSIF mudam de escala ao longo de 2019–2024.2. **Z-score dentro de (segmento, trimestre)**, com mediana e MAD — a pergunta é "esta instituição   destoa dos pares *agora*?". Sem esse passo, o modelo aprende sobretudo tamanho e ciclo econômico.3. **Normalização ajustada apenas no treino** — o contrário é vazamento.4. **Ausentes marcados, não imputados pela média** — cada atributo ganha uma coluna-sombra.   Imputar a média faria a ausência parecer normalidade e reduziria o erro de reconstrução   justamente onde há menos informação.Sobre a partição: **corte cronológico**, não aleatório. Painel contábil é altamente persistentedentro da instituição; uma divisão aleatória 70/30 coloca trimestres futuros no treino e passadosno teste, e o modelo passa a "reconhecer a instituição" em vez de generalizar. A seção 8 quantificaesse efeito.

In [ ]:
pronto = painel.construir(ind)
display(pronto.resumo())
print(f"\nAtributos de entrada: {len(pronto.atributos)}")
print("Observacoes por instituicao (mediana):",
      int(pronto.dados.groupby('cod_inst').size().median()))

---## 8. Auditoria de vazamentoDuas verificações que decidem se os resultados podem ser reportados.**Vazamento temporal.** A partição respeita a ordem do tempo?**Circularidade.** Este é o ponto crítico da replicação. Benyasrisawat e Ounlert (2026) treinamXGBoost para prever o DF-Score usando como atributos os **próprios componentes do DF-Score** —que, por definição (Tabela 3 do artigo), é a soma deles. A acurácia de 0,995 e o AUC-ROC de 0,999reportados na Tabela 5 são compatíveis com um modelo que reaprende uma identidade aritmética quelhe foi dada de entrada, não com um modelo que detecta fraude.Isso não invalida o artigo: o objeto dele é a relação entre risco de fraude e opinião de auditoria,estimada depois por IV-2SLS. Invalida, sim, a leitura de que "AUC 0,999 mostra capacidade dedetecção" — e replicar o desenho sem essa observação transportaria o artefato para o TCC.

In [ ]:
auditoria = vazamento.auditar_particao_temporal(pronto.dados, pronto.treino, pronto.teste)
for k, v in auditoria.items():
    print(f"  {k:32s} {v}")

---## 9. Rede neural — autoencoder denoising**Por que autoencoder e não classificador.** Não existe rótulo público de fraude para bancosbrasileiros. Sem rótulo, um classificador precisa de um alvo inventado — e a métrica passa a medirconcordância com a invenção. O autoencoder dispensa rótulo: aprende a variedade de comportamentonormal do conjunto e erra mais onde a observação não pertence a ela. O erro de reconstrução é oescore.**Por que capacidade pequena.** S2+S3 tem ordem de dezenas de instituições × 24 trimestres. Umarede larga memoriza o painel, reconstrói bem também as observações atípicas, e o escore deixa dediscriminar. O pipeline monitora a razão parâmetros/observações e emite pendência quando elaextrapola.**Por que várias sementes.** O escore final é a mediana sobre sementes, e a correlação de postoentre elas mede confiabilidade. Abaixo de ~0,80 o ranking depende mais da inicialização do que dosdados — e então nenhuma instituição individual pode ser apontada.

In [ ]:
todos = np.ones(len(pronto.dados), dtype=bool)
X_tr, X_va, X_todos = pronto.matriz(pronto.treino), pronto.matriz(pronto.validacao), pronto.matriz(todos)

resultado = redes.treinar_autoencoder(X_tr, X_va, X_todos, verboso=True)

print(f"\nEpocas efetivas por semente : {resultado.epocas_efetivas}")
print(f"Perda de validacao (melhor) : {np.round(resultado.perda_validacao, 5)}")
print(f"Estabilidade entre sementes : {resultado.estabilidade:.3f}  (Spearman medio)")

In [ ]:
escores = pronto.dados[["cod_inst", "anomes", "segmento"]].reset_index(drop=True).copy()
escores["escore_rede"] = resultado.escores
escores["posto"] = escores["escore_rede"].rank(pct=True)

fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
eixos[0].hist(resultado.escores, bins=50, color="#4C6EF5", edgecolor="white")
eixos[0].axvline(np.quantile(resultado.escores, CONFIG.avaliacao.percentil_alerta),
                 color="#E03131", ls="--", label=f"percentil {CONFIG.avaliacao.percentil_alerta:.0%}")
eixos[0].set_title("Distribuicao do erro de reconstrucao"); eixos[0].legend()

for seg, g in escores.groupby("segmento"):
    serie = g.groupby("anomes")["escore_rede"].median()
    eixos[1].plot(serie.index.astype(str), serie.values, marker="o", label=seg)
eixos[1].set_title("Mediana do escore por trimestre e segmento")
eixos[1].tick_params(axis="x", rotation=60); eixos[1].legend()
plt.tight_layout(); plt.show()

---## 10. Rede sequencial (GRU): quebra estruturalO autoencoder da seção 9 é transversal: compara a instituição com os pares no mesmo trimestre. OGRU faz outra pergunta — lê a janela de trimestres anteriores da própria instituição e prevê oseguinte. O erro de previsão mede **mudança de comportamento**, que é mais próximo da intuição deauditoria ("esta instituição passou a operar de outro jeito") do que a atipicidade transversal.Os dois escores são complementares e devem ser reportados separadamente: uma instituição pode sersempre diferente dos pares sem nunca mudar (alto no autoencoder, baixo no GRU), ou mudarbruscamente permanecendo dentro do intervalo dos pares (o inverso).

In [ ]:
Xs, ys, idx_seq = painel.sequencias(pronto, janela=CONFIG.painel.janela_temporal)
print("Sequencias formadas:", Xs.shape)

if len(Xs):
    m_tr = (idx_seq["anomes"] <= CONFIG.avaliacao.corte_treino).to_numpy()
    m_va = ((idx_seq["anomes"] > CONFIG.avaliacao.corte_treino) &
            (idx_seq["anomes"] <= CONFIG.avaliacao.corte_validacao)).to_numpy()
    res_seq = redes.treinar_sequencial(Xs[m_tr], ys[m_tr], Xs[m_va], ys[m_va], Xs, ys)
    seq = idx_seq.copy(); seq["escore_gru"] = res_seq.escores
    escores = escores.merge(seq, on=["cod_inst", "anomes"], how="left")
    print(f"Estabilidade entre sementes (GRU): {res_seq.estabilidade:.3f}")
    comum = escores.dropna(subset=["escore_gru"])
    print("Correlacao de posto autoencoder x GRU:",
          round(comum["escore_rede"].corr(comum["escore_gru"], method="spearman"), 3))

---## 11. A rede acrescenta algo? Comparação com métodos mais simplesSem baseline, "a rede encontrou 12 instituições atípicas" é uma lista, não um resultado. A perguntaque a banca fará é se a rede supera métodos mais simples e mais auditáveis.- **PCA** — erro de reconstrução linear. Se o autoencoder não superar o PCA, a não linearidade não  está comprando nada neste painel.- **Mahalanobis robusto** — distância estatística clássica, com covariância de determinante mínimo  (a covariância comum é estimada com os próprios pontos atípicos dentro, o que mascara a anomalia).- **Isolation Forest** — método de árvore, sem suposição de distribuição.**Como ler a matriz de concordância.** Correlação alta com o PCA significa que a rede estáredescobrindo o que um método linear já dizia — e aí o custo de interpretabilidade não se justifica.Correlação baixa exige explicar, caso a caso, *o que* a rede enxerga a mais. As duas leituras sãoresultados publicáveis; a que não é aceitável é não fazer a comparação.

In [ ]:
comparacao = baselines.painel_de_escores(X_tr, X_todos, escore_rede=resultado.escores)
concordancia = baselines.concordancia(comparacao)
display(concordancia.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(concordancia.values, cmap="RdYlBu_r", vmin=-1, vmax=1)
rotulos_curtos = [c.replace("posto_", "") for c in concordancia.columns]
ax.set_xticks(range(len(rotulos_curtos)), rotulos_curtos, rotation=45, ha="right")
ax.set_yticks(range(len(rotulos_curtos)), rotulos_curtos)
for i in range(len(rotulos_curtos)):
    for j in range(len(rotulos_curtos)):
        ax.text(j, i, f"{concordancia.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Concordancia de posto entre metodos"); plt.colorbar(im); plt.tight_layout(); plt.show()

---## 12. Interpretação: de "atípico" para "atípico em quê"Um escore sem explicação é inútil para auditoria. Ninguém abre procedimento com "o modelo deu 0,93".A decomposição do erro do autoencoder é **exata**, não estimada: o erro total é a soma dos erros poratributo. Essa é a principal vantagem prática do autoencoder sobre modelos de árvore com SHAP nestecontexto — o auditor recebe a atribuição sem uma camada adicional de aproximação.Se o topo do ranking de atributos for dominado pelos indicadores prudenciais propostos (provisão,Basileia, crédito tributário), o trabalho ganha o argumento de que **a adaptação ao contextobancário foi o que produziu informação** — e não a transposição dos modelos de empresa nãofinanceira. Esse é o achado metodológico mais forte que este desenho pode entregar.

In [ ]:
indice = pronto.dados[["cod_inst", "anomes"]].reset_index(drop=True)
decomposicao = interpretacao.decompor_erro(resultado.erro_por_atributo, pronto.atributos, indice, top=5)
ranking = interpretacao.ranking_atributos(decomposicao)
display(ranking.head(15))

catalogo = indicadores.catalogo_df()
classe = dict(zip(catalogo["indicador"], catalogo["evidencia"]))
ranking["classe_evidencia"] = (ranking["atributo"]
                               .str.replace("z_par_", "", regex=False)
                               .str.replace("faltante_", "", regex=False)
                               .map(classe))
print("\nParticipacao por classe de evidencia (quanto do topo vem de cada origem):")
display(ranking.groupby("classe_evidencia")["vezes_no_topo"].sum().sort_values(ascending=False))

In [ ]:
topo = escores.nlargest(10, "escore_rede")
display(topo[["cod_inst", "anomes", "segmento", "escore_rede", "posto"]])

for _, r in topo.head(3).iterrows():
    print("\n" + interpretacao.narrativa(decomposicao, r["cod_inst"], r["anomes"], catalogo))

---## 13. Triangulação independente: lei de Newcomb–BenfordRede, PCA e Mahalanobis derivam todos do **mesmo** conjunto de indicadores. Se os três apontarem amesma instituição, isso não é corroboração — é a mesma medida contada três vezes.Benford opera sobre outra coisa: a distribuição dos **dígitos** dos saldos publicados,independentemente de qualquer razão contábil construída aqui. Concordância entre um escore alto derede e não conformidade a Benford é evidência genuinamente independente.Três ressalvas que precisam constar no texto: (i) saldos do IF.data já são agregados, e aconformidade a Benford é mais fraca em agregados do que em lançamentos individuais; (ii) contas compiso, teto ou valor regulado violam a premissa de escala livre; (iii) desvio de Benford **não éindício de fraude** — arredondamento, conversão e reclassificação produzem o mesmo sintoma.

In [ ]:
if USAR_DADOS_REAIS and RELATORIOS:
    longo_benford = bruto_longo.rename(columns={COL_INST: "cod_inst", COL_VALOR: "saldo"})
    conformidade = benford.por_instituicao(longo_benford, "saldo", minimo_observacoes=300)
    display(conformidade.head(15))

    r = benford.testar(longo_benford["saldo"].to_numpy(dtype=float), posicao=1)
    if r["n"]:
        fig, ax = plt.subplots(figsize=(7, 3.5))
        largura = 0.38
        ax.bar(r["digitos"] - largura/2, r["observado"], largura, label="observado", color="#4C6EF5")
        ax.bar(r["digitos"] + largura/2, r["esperado"], largura, label="esperado (Benford)", color="#ADB5BD")
        ax.set_xticks(r["digitos"]); ax.set_xlabel("primeiro digito significativo")
        ax.set_title(f"Conformidade agregada — MAD={r['mad']:.4f} ({r['classificacao']})")
        ax.legend(); plt.tight_layout(); plt.show()

### 13.2 Os seis testes analíticos

Mesma lógica de triangulação da lei de Newcomb–Benford: um resultado que **não depende da rede** —
nem de treino, partição, semente ou convergência — e que por isso é reproduzível por qualquer
terceiro a partir da base bruta.

Um teste por parâmetro contábil obrigatório, cada um isolando um mecanismo de manipulação
conhecido. Já foram executados por `processar_base.py`; as células abaixo apenas lêem e exibem.
Documentação completa em `docs/TESTES_ANALITICOS.md`.

**Leitura obrigatória antes de citar qualquer número daqui:** sinalização não é indício de
irregularidade. É divergência frente ao comportamento dos pares, que constitui hipótese a ser
examinada. Mudança de modelo de negócio, incorporação e alteração de política contábil produzem o
mesmo sintoma.

In [ ]:
display(testes_analiticos.catalogo_df()[["teste", "nome", "parametro_ancora", "hipotese"]])

sinal = testes_analiticos.aplicar(ind)
print("\nSinalizacoes por teste:")
display(testes_analiticos.resumo(sinal))

#### Sensibilidade ao limiar

O limiar é parâmetro de calibração da carga de exame, **não nível de significância**: sob
normalidade 2,5 desvios dariam ~1% por cauda, proporção que não se verifica nestas razões, de
cauda pesada e concentradas em zero. Reportar sempre o limiar adotado junto com a taxa que ele
produz, e esta tabela.

In [ ]:
display(testes_analiticos.sensibilidade(ind))

#### Estrutural ou episódico

A comparação com os pares mede **nível**, e nível persistentemente distinto reflete o modelo de
negócio, não uma ocorrência. Uma instituição que destoa em 2 de 24 trimestres oferece uma pergunta
datada, confrontável com fatos do período; uma que destoa em 24 de 24 apenas informa que opera em
outro segmento de mercado. É o `foco_de_exame` que se leva para procedimento.

In [ ]:
persistencia = testes_analiticos.classificar_persistencia(sinal)
display(pd.crosstab(persistencia["teste"], persistencia["classe"]))

foco = testes_analiticos.foco_de_exame(sinal)
print(f"Foco de exame: {len(foco):,} observacoes de {foco['cod_inst'].nunique()} instituicoes")
display(testes_analiticos.resumo(foco))

# Texto auditavel de uma observacao concreta
if len(foco):
    alvo = foco.nlargest(1, "n_testes_sinalizados").iloc[0]
    print("\n" + testes_analiticos.detalhar(foco, alvo["cod_inst"], alvo["anomes"]))

---## 14. Validação por eventos públicos**A seção que decide o alcance da conclusão do trabalho.**Sem eventos, o TCC só pode afirmar "identifiquei instituições contabilmente atípicas". Com eventosverificáveis, pode afirmar algo bem mais forte: "o escore subiu *N* trimestres antes de eventosque efetivamente ocorreram".Preencha `dados/externos/eventos_publicos.csv` com eventos **datados e verificáveis em fonteprimária**: atos do Banco Central de liquidação extrajudicial, regime de administração especialtemporária, intervenção, cancelamento de autorização; processos administrativos sancionadores;fatos relevantes na CVM. A coluna `fonte` deve trazer a URL ou a referência do documento — é o quepermite à banca conferir.Duas advertências de desenho, ambas incorporadas ao código:- Trimestres **posteriores** ao evento são excluídos. Depois da intervenção o balanço reflete a  intervenção; acertar ali não é previsão, é tautologia.- Com poucos positivos, `Precision@k` tem intervalo de confiança largo demais para sustentar  comparação entre modelos. O honesto é apresentar como **estudo de caso**, e o pipeline emite  pendência nesse sentido.A métrica pertinente aqui não é AUC-ROC, e sim precisão no topo da lista: o auditor tem recursofinito e vai examinar as *k* primeiras. AUC-ROC integra sobre faixas da curva que ninguémexaminaria.

In [ ]:
caminho_eventos = DIR_EXTERNOS / "eventos_publicos.csv"
if caminho_eventos.exists():
    eventos = pd.read_csv(caminho_eventos, dtype={"cod_inst": str})
    eventos = eventos[eventos["anomes_evento"].notna()]
    print(f"{len(eventos)} evento(s) informado(s).")
    display(eventos)
else:
    eventos = pd.DataFrame(columns=["cod_inst", "anomes_evento", "tipo_evento", "fonte"])
    print("Arquivo de eventos ausente — ver modelo em dados/externos/eventos_publicos_MODELO.csv")

validacao = avaliacao.validar_por_eventos(escores.rename(columns={"escore_rede": "escore"}), eventos)
if not validacao.tabela_k.empty:
    display(validacao.tabela_k)
    display(validacao.antecedencia)

In [ ]:
persistencia = avaliacao.estabilidade_temporal(escores.rename(columns={"escore_rede": "escore"}))
print("Persistencia trimestral media (Spearman):", round(persistencia["spearman"].mean(), 3))
display(persistencia.tail(8))

---## 15. Comparação com XGBoost e LightGBM, sob guarda de circularidadeO projeto de pesquisa previa comparar XGBoost e LightGBM. Eles permanecem — agora como **termo decomparação** da rede, e sob a restrição da seção 8.O procedimento tem duas rodadas deliberadas:1. **Com** os componentes do rótulo entre os atributos — reproduz o desenho do artigo de referência   e deve produzir métrica quase perfeita.2. **Sem** eles — o resultado honesto.A diferença entre as duas é, ela própria, um achado do TCC: quantifica quanto da performancereportada em desenhos desse tipo vem de circularidade e não de capacidade preditiva.O rótulo aqui é **fraco** — construído pelo autor a partir dos indicadores. Toda métrica calculadacontra ele mede concordância com uma regra do autor, e assim precisa ser nomeada no texto:*"concordância com o rótulo fraco"*, jamais *"detecção de fraude"*.

In [ ]:
COMPONENTES_ROTULO = ["levi_bancario", "aqi_bancario", "accrual_bancario"]
y_fraco, componentes = avaliacao.rotulo_fraco(ind, COMPONENTES_ROTULO, percentil=0.90)
print(f"Rotulo fraco: {y_fraco.sum()} positivos em {len(y_fraco)} ({y_fraco.mean():.1%})")

guarda = vazamento.GuardaCircularidade(componentes_do_rotulo=componentes, nome_do_rotulo="rotulo_fraco")
atributos_limpos = guarda.filtrar(pronto.atributos)
print(f"Atributos: {len(pronto.atributos)} -> {len(atributos_limpos)} apos remover circulares")

In [ ]:
alvo = pronto.dados[["cod_inst", "anomes"]].merge(
    ind[["cod_inst", "anomes"]].assign(y=y_fraco), on=["cod_inst", "anomes"], how="left")["y"].fillna(0).to_numpy()

resultados_gbm = {}
for rotulo_teste, atributos in (("com componentes (circular)", pronto.atributos),
                                ("sem componentes (honesto)", atributos_limpos)):
    Xc = pronto.dados[atributos].to_numpy(dtype=np.float32)
    for lib in ("xgboost", "lightgbm"):
        modelo, prob = baselines.treinar_gbm(Xc[pronto.treino], alvo[pronto.treino], Xc[pronto.teste], lib)
        if modelo is not None:
            resultados_gbm[f"{lib} — {rotulo_teste}"] = avaliacao.metricas_classificacao(
                alvo[pronto.teste], prob)

if resultados_gbm:
    display(pd.DataFrame(resultados_gbm).T.round(3))

In [ ]:
auditoria_corr = guarda.auditar_correlacao(
    pd.DataFrame(pronto.dados[atributos_limpos]), alvo)
display(auditoria_corr.head(10).round(3))

---## 16. PONTOS DE CORREÇÃOConsolidação de tudo que ficou fora de alcance ou incerto durante a execução, com a **árearesponsável** e a **ação pedida**.Cada item foi levantado pelo próprio código no momento em que a limitação apareceu — não é umalista redigida depois. Itens marcados `BLOQUEIA RESULTADO` significam que os escores desta execução**não devem ser reportados como resultado no TCC** enquanto não forem resolvidos.Uma observação para a leitura: ausência de pendências não equivale a validação metodológica.Significa apenas que nenhuma verificação automatizada falhou. As decisões substantivas —conglomerado ou instituição individual, quais eventos públicos entram, se os indicadores propostossão defensáveis — continuam sendo do pesquisador e do orientador.

In [ ]:
print(diagnostico.relatorio())

In [ ]:
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
pendencias = diagnostico.para_dataframe()
if not pendencias.empty:
    pendencias.to_csv(DIR_RESULTADOS / "pontos_de_correcao.csv", index=False)

if USAR_DADOS_REAIS:
    escores.to_csv(DIR_RESULTADOS / "escores_atipicidade.csv", index=False)
    decomposicao.to_csv(DIR_RESULTADOS / "decomposicao_erro.csv", index=False)
    ranking.to_csv(DIR_RESULTADOS / "ranking_atributos.csv", index=False)
    indicadores.catalogo_df().to_csv(DIR_RESULTADOS / "quadro_operacionalizacao.csv", index=False)
    print("Resultados gravados em", DIR_RESULTADOS)
else:
    print("MODO TESTE: nada foi gravado como resultado (correto — nao sao dados).")

---## Encerramento: o que este notebook autoriza afirmar**Autoriza:**- que determinadas instituições-trimestre se afastam do padrão dos pares e do próprio histórico,  com a decomposição indicando em quais contas;- que o afastamento é (ou não é) estável entre sementes, no tempo e entre métodos;- que a rede supera (ou não) PCA, Mahalanobis e Isolation Forest no mesmo painel — resultado  publicável nas duas direções;- que os indicadores prudenciais adaptados carregam (ou não) mais informação que a transposição  direta dos modelos de empresa não financeira;- quanto da performance de um desenho supervisionado neste domínio vem de circularidade.**Não autoriza:**- afirmar que qualquer instituição cometeu fraude, irregularidade ou manipulação;- tratar o escore como prova, indício jurídico ou opinião de auditoria;- inferir causalidade entre indicador e conduta;- generalizar para instituições fora de S2/S3 ou fora do recorte 2019–2024.A conclusão defensável tem esta forma: *modelos não lineares aplicados a dados públicos doIF.data/COSIF produzem uma ordenação de risco que concentra, nas primeiras posições, instituiçõescom características contábeis que a literatura associa a maior probabilidade de manipulação —oferecendo à auditoria um critério de priorização que os modelos lineares tradicionais, calibradospara empresas não financeiras, não fornecem para este segmento.*